In [1]:
from __future__ import annotations

import lhapdf

import numpy as np

from validphys.api import API
from validphys.fkparser import load_fktable
from validphys.loader import Loader
import matplotlib.pyplot as plt


# Generate data

## Theory ID 208

The same as in "T3_Beta.py"


In [2]:
inp_p = {
    "dataset_input": {"dataset": "BCDMS_NC_NOTFIXED_P_EM-F2", "variant": "legacy"},
    "use_cuts": "internal",
    "theoryid": 208, #208
}
inp_d = {
    "dataset_input": {"dataset": "BCDMS_NC_NOTFIXED_D_EM-F2", "variant": "legacy"},
    "use_cuts": "internal",
    "theoryid": 208,
}

lcd_p = API.loaded_commondata_with_cuts(**inp_p)
lcd_d = API.loaded_commondata_with_cuts(**inp_d)

df_p = (
    lcd_p.commondata_table.reset_index()
    .rename(
        columns={
            "kin1": "x",
            "kin2": "q2",
            "kin3": "y",
            "data": "F2_p",
            "stat": "error",
            "entry": "entry_p",
        },
    )
    .assign(idx_p=lambda df: df.index)
)
df_d = (
    lcd_d.commondata_table.reset_index()
    .rename(
        columns={
            "kin1": "x",
            "kin2": "q2",
            "kin3": "y",
            "data": "F2_d",
            "stat": "error",
            "entry": "entry_d",
        },
    )
    .assign(idx_d=lambda df: df.index)
)

# Merge on (x, q2) to form F2_p - F2_d
mp = 0.938
mp2 = mp**2

#modify merge function to remove double counting
merged_df = df_p.merge(df_d, on=["x", "q2"], suffixes=("_p", "_d")).assign(
    y_val=lambda df: (df["F2_p"] - df["F2_d"]),
    F2_d = lambda df: (df["F2_d"]),
    w2=lambda df: df["q2"] * (1 - df["x"]) / df["x"] + mp2,
)

# remove duplicates to match 248 entries as in paper, for full 611 entries just comment out this line
merged_df = merged_df.groupby(['x', 'q2', 'F2_d', 'entry_d']).first().reset_index()

# Extract q2_vals and y_real for later use
q2_vals = merged_df["q2"].to_numpy()
y_real = merged_df["y_val"].to_numpy()
np.save("Dataset/q2_vals.npy", q2_vals)
np.save("Dataset/y_real.npy", y_real)

In [3]:
# This cell was just for bookkeeping double counting indices

entry_p_rel = merged_df["entry_p"].to_numpy() - 1
entry_d_rel = merged_df["entry_d"].to_numpy() - 1
print(entry_d_rel)

q2_vals = merged_df["q2"].to_numpy()

print(len(q2_vals))


[  0   1  99 100   2   3   4 101 102 103 178 179   5   6   7   8   9 104
 105 106 107 180 181 182 183  10  11  12  13  14  15 108  16 109 110 111
 112 184 113 185 186 187 188  17  18  19  20  21  22  23  24 114  25 115
 189 116 117 190 118 191 119 192 193 194 195  26  27  28  29  30  31  32
 120  33 121 196  34 122  35 123 197 198 124 199 125 126 200 201 127 202
 203 204  36  37  38  39  40  41  42  43 128 129  44  45 130 205  46 131
 206  47 207 132 133 208 209 134 210 135 136 211 212 213  48  49  50  51
  52  53  54 137  55 138  56 214  57 139  58 215 140 216  59 141  60 217
 142  61 218 143 144 219 145 220 146 221 222 223  63  64  65  66  67 147
  68 148  69 224 149  70 150  71 225  72 151 226 227  73 152  74 153 228
  75 229 154 230 155 231 156 232 157 233 234  79  80  81 158  82 159 160
  83  84 161 235  85 162 236  86 237 163 164 238  87  88 165 239 166 240
  89 241 167 168 242 243 244  92 169  93 170 171  94 245 172  95 246  96
 247 173  97 174 248  98 249 175 250 176 251 177 25

In [23]:
print(len(y_real))
t3_index = 2  # flavor index in FK table
loader = Loader()
fk_p = load_fktable(loader.check_fktable(setname="BCDMSP", theoryID=208, cfac=()))
fk_d = load_fktable(loader.check_fktable(setname="BCDMSD", theoryID=208, cfac=()))

wp = fk_p.get_np_fktable()  # shape (n_data_fk, n_flav, n_grid)
wd = fk_d.get_np_fktable()
wp_t3 = wp[:, t3_index, :]
wd_t3 = wd[:, t3_index, :]

entry_p_rel = merged_df["entry_p"].to_numpy() - 1
entry_d_rel = merged_df["entry_d"].to_numpy() - 1
W = wp_t3[entry_p_rel] - wd_t3[entry_d_rel]  # shape (n_data, n_grid)

print(W.shape)

# Save xgrid for later normalization
xgrid = fk_p.xgrid.copy()  # shape (n_grid,)

print(fk_p.xgrid.shape)

# Save the W matrix for later use
np.save("Dataset/W.npy", W)
np.save("Dataset/xgrid.npy", xgrid)

248
(248, 50)
(50,)


In [8]:
params_cov = {
    "dataset_inputs": [inp_p["dataset_input"], inp_d["dataset_input"]],
    "use_cuts": "internal",
    "theoryid": 208,
}
cov_full = API.dataset_inputs_covmat_from_systematics(**params_cov)

# Suppose merged_df has columns idx_p and idx_d (these were created earlier in your preprocessing)
idx_p_merge = merged_df["idx_p"].to_numpy()  # length = N (number of matched points)
idx_d_merge = merged_df["idx_d"].to_numpy()  # length = N (same N)

# cov_full is (Np + Nd) x (Np + Nd), so:
n_p = len(df_p)
# Extract the proton-proton, deuteron-deuteron, and proton-deuteron sub-blocks:
c_pp = cov_full[:n_p, :n_p]  # shape = (Np, Np)
c_dd = cov_full[n_p:, n_p:]  # shape = (Nd, Nd)
c_pd = cov_full[:n_p, n_p:]  # shape = (Np, Nd)

# Now restrict each block to only those rows/cols that appear in merged_df:
c_pp_sub = c_pp[np.ix_(idx_p_merge, idx_p_merge)]  # (N, N)
c_dd_sub = c_dd[np.ix_(idx_d_merge, idx_d_merge)]  # (N, N)
c_pd_sub = c_pd[np.ix_(idx_p_merge, idx_d_merge)]  # (N, N)


c_yy = c_pp_sub + c_dd_sub - 2 * c_pd_sub

# Make sure it's exactly symmetric:
c_yy = 0.5 * (c_yy + c_yy.T)

print(c_yy.shape)

# Add jitter until positive-definite
jitter = 1e-6 * np.mean(np.diag(c_yy))
for _ in range(10):
    try:
        np.linalg.cholesky(c_yy)
        break
    except np.linalg.LinAlgError:
        c_yy += np.eye(c_yy.shape[0]) * jitter
        jitter *= 10
else:
    msg = "Covariance matrix not positive-definite"
    raise RuntimeError(msg)

np.save('Dataset/c_yy.npy', c_yy)

(248, 248)


In [18]:
#Compute reference for closure test, this should be the f0/y0
pdfset = lhapdf.getPDFSet("NNPDF40_nnlo_as_01180")  # This PDF can be changed to any toy underlying PDF set
pdf0 = pdfset.mkPDF(0)
Q0 = fk_p.Q0
xt3_true = np.zeros_like(xgrid)


# T_3 = (u - ubar) - (d - dbar)
for i, x in enumerate(xgrid):
    u = pdf0.xfxQ(2, x, Q0)
    ub = pdf0.xfxQ(-2, x, Q0)
    d = pdf0.xfxQ(1, x, Q0)
    db = pdf0.xfxQ(-1, x, Q0)
    xt3_true[i] = (u - ub) - (d - db)


t3 = xt3_true / xgrid # Or here we can just input the true function directly for T3 or xT3

t3_ref_int = np.trapz(xt3_true / xgrid, xgrid)  # noqa: NPY201


y_theory = W @ (xt3_true)  # shape (N,)
print(xt3_true)
y_t3_theory = W @ (t3)  # shape (N,)

np.save('Dataset/y_theory.npy', y_theory)
np.save('Dataset/y_t3_theory.npy', y_t3_theory)
np.save('Dataset/xt3_true.npy', xt3_true)
np.save('Dataset/t3_true.npy', t3)
print(len(y_theory))

rng = np.random.default_rng(seed=451)  # you can set seed if you want reproducible “data”
noise = rng.multivariate_normal(mean=np.zeros(len(y_theory)), cov=c_yy)

y_pseudo = y_theory + noise
#y_theory: y0, y_pseudo = y

LHAPDF 6.5.5 loading /opt/homebrew/Caskroom/miniconda/base/envs/nnpdf/share/LHAPDF/NNPDF40_nnlo_as_01180/NNPDF40_nnlo_as_01180_0000.dat
[ 2.23406609e-01  2.38743349e-01  2.54108287e-01  2.69389000e-01
  2.84450202e-01  2.99145606e-01  3.13306148e-01  3.26738388e-01
  3.39234895e-01  3.50576292e-01  3.60542697e-01  3.68925595e-01
  3.75539512e-01  3.80230998e-01  3.82883849e-01  3.83421959e-01
  3.81809307e-01  3.78047017e-01  3.72171051e-01  3.64248175e-01
  3.54374553e-01  3.42672712e-01  3.29288043e-01  3.14386276e-01
  2.98146616e-01  2.80758879e-01  2.62411805e-01  2.43292305e-01
  2.23587652e-01  2.03496044e-01  1.83227191e-01  1.63009184e-01
  1.43082874e-01  1.23705184e-01  1.05137815e-01  8.76371308e-02
  7.14455275e-02  5.67757901e-02  4.38025527e-02  3.26523955e-02
  2.33876599e-02  1.59916393e-02  1.03547516e-02  6.27677277e-03
  3.49626299e-03  1.73098823e-03  7.12424876e-04  2.05713383e-04
  1.63475678e-05 -1.19352590e-05]
NNPDF40_nnlo_as_01180 PDF set, member #0, version 

In [10]:
# generate test data
test_xgrid = rng.uniform(0, 1, size=20)

pdfset = lhapdf.getPDFSet("NNPDF40_nnlo_as_01180")  # This PDF can be changed to any toy underlying PDF set
pdf0 = pdfset.mkPDF(0)
Q0 = fk_p.Q0
xt3_test = np.zeros_like(test_xgrid)


# T_3 = (u - ubar) - (d - dbar)
for i, x in enumerate(test_xgrid):
    u = pdf0.xfxQ(2, x, Q0)
    ub = pdf0.xfxQ(-2, x, Q0)
    d = pdf0.xfxQ(1, x, Q0)
    db = pdf0.xfxQ(-1, x, Q0)
    xt3_test[i] = (u - ub) - (d - db)

t3_test= xt3_test / test_xgrid 

np.save('Dataset/xt3_test.npy', xt3_test)
np.save('Dataset/xgrid_test.npy', test_xgrid)

LHAPDF 6.5.5 loading /opt/homebrew/Caskroom/miniconda/base/envs/nnpdf/share/LHAPDF/NNPDF40_nnlo_as_01180/NNPDF40_nnlo_as_01180_0000.dat
NNPDF40_nnlo_as_01180 PDF set, member #0, version 1; LHAPDF ID = 331100


## Theory ID 40001000

In [11]:
inp_p2 = {
    "dataset_input": {"dataset": "BCDMS_NC_NOTFIXED_P_EM-F2", "variant": "legacy"},
    "use_cuts": "internal",
    "theoryid": 40001000, #208
}
inp_d2 = {
    "dataset_input": {"dataset": "BCDMS_NC_NOTFIXED_D_EM-F2", "variant": "legacy"},
    "use_cuts": "internal",
    "theoryid": 40001000,
}

lcd_p2 = API.loaded_commondata_with_cuts(**inp_p2)
lcd_d2 = API.loaded_commondata_with_cuts(**inp_d2)

df_p2 = (
    lcd_p2.commondata_table.reset_index()
    .rename(
        columns={
            "kin1": "x",
            "kin2": "q2",
            "kin3": "y",
            "data": "F2_p",
            "stat": "error",
            "entry": "entry_p",
        },
    )
    .assign(idx_p=lambda df: df.index)
)
df_d2 = (
    lcd_d2.commondata_table.reset_index()
    .rename(
        columns={
            "kin1": "x",
            "kin2": "q2",
            "kin3": "y",
            "data": "F2_d",
            "stat": "error",
            "entry": "entry_d",
        },
    )
    .assign(idx_d=lambda df: df.index)
)

# Merge on (x, q2) to form F2_p - F2_d
mp = 0.938
mp2 = mp**2

#modify merge function to remove double counting
merged_df2 = df_p2.merge(df_d2, on=["x", "q2"], suffixes=("_p", "_d")).assign(
    y_val=lambda df: (df["F2_p"] - df["F2_d"]),
    F2_d = lambda df: (df["F2_d"]),
    w2=lambda df: df["q2"] * (1 - df["x"]) / df["x"] + mp2,
)

# remove duplicates to match 248 entries as in paper, for full 611 entries just comment out this line
merged_df2 = merged_df2.groupby(['x', 'q2', 'F2_d', 'entry_d']).first().reset_index()

# Extract q2_vals and y_real for later use
q2_vals2 = merged_df2["q2"].to_numpy()
y_real2 = merged_df2["y_val"].to_numpy()
np.save("Dataset/q2_vals_newth.npy", q2_vals2)
np.save("Dataset/y_real_newth.npy", y_real2)

In [12]:
entry_p_rel2 = merged_df2["entry_p"].to_numpy() - 1
entry_d_rel2 = merged_df2["entry_d"].to_numpy() - 1
print(entry_d_rel2)

q2_vals2 = merged_df2["q2"].to_numpy()

print(len(q2_vals2))

[  0   1  99 100   2   3   4 101 102 103 178 179   5   6   7   8   9 104
 105 106 107 180 181 182 183  10  11  12  13  14  15 108  16 109 110 111
 112 184 113 185 186 187 188  17  18  19  20  21  22  23  24 114  25 115
 189 116 117 190 118 191 119 192 193 194 195  26  27  28  29  30  31  32
 120  33 121 196  34 122  35 123 197 198 124 199 125 126 200 201 127 202
 203 204  36  37  38  39  40  41  42  43 128 129  44  45 130 205  46 131
 206  47 207 132 133 208 209 134 210 135 136 211 212 213  48  49  50  51
  52  53  54 137  55 138  56 214  57 139  58 215 140 216  59 141  60 217
 142  61 218 143 144 219 145 220 146 221 222 223  63  64  65  66  67 147
  68 148  69 224 149  70 150  71 225  72 151 226 227  73 152  74 153 228
  75 229 154 230 155 231 156 232 157 233 234  79  80  81 158  82 159 160
  83  84 161 235  85 162 236  86 237 163 164 238  87  88 165 239 166 240
  89 241 167 168 242 243 244  92 169  93 170 171  94 245 172  95 246  96
 247 173  97 174 248  98 249 175 250 176 251 177 25

In [14]:
import numpy as np
from validphys.api import API
from validphys.pineparser import pineappl_reader

theoryid = 40001000
t3_index = 2  # flavor index in FK table

# Load datasets (legacy names OK; they get mapped internally)
ds_p = API.dataset(dataset_input={"dataset": "BCDMS_NC_NOTFIXED_P_EM-F2"}, theoryid=theoryid, use_cuts="internal")
ds_d = API.dataset(dataset_input={"dataset": "BCDMS_NC_NOTFIXED_D_EM-F2"}, theoryid=theoryid, use_cuts="internal")

# Read PineAPPL FK tables for each dataset (often multiple sqrt(s) grids)
fk_p_list = [pineappl_reader(fkspec) for fkspec in ds_p.fkspecs]
fk_d_list = [pineappl_reader(fkspec) for fkspec in ds_d.fkspecs]

# If you truly need one combined FK table like the legacy case, you must pick the
# matching component for each datapoint. A simple starting point is to use the first.
fk_p2 = fk_p_list[0]
fk_d2 = fk_d_list[0]

wp2 = fk_p2.get_np_fktable()  # (n_data_fk, n_flav, n_grid)
wd2 = fk_d2.get_np_fktable()

wp2_t3 = wp2[:, t3_index, :]
wd2_t3 = wd2[:, t3_index, :]

entry_p_rel2 = merged_df2["entry_p"].to_numpy() - 1
entry_d_rel2 = merged_df2["entry_d"].to_numpy() - 1

W2 = wp2_t3[entry_p_rel2] - wd2_t3[entry_d_rel2]  # (n_data, n_grid)
print(W2.shape)

# Save xgrid for later normalization
xgrid2 = fk_p2.xgrid.copy()  # (n_grid,)
print(fk_p2.xgrid.shape)
# Save for later use
np.save("Dataset/W_newth.npy", W2)
np.save("Dataset/xgrid_newth.npy", xgrid2)


(248, 27)
(27,)


In [15]:
import numpy as np
from validphys.api import API

theoryid = 40001000

params_cov2 = {
    "dataset_inputs": [
        {"dataset": "BCDMSP"},  # legacy name OK
        {"dataset": "BCDMSD"},
    ],
    "use_cuts": "internal",
    "theoryid": theoryid,
}

# (Np+Nd) x (Np+Nd) covariance for the concatenated (post-cut) inputs
cov_full2 = API.dataset_inputs_covmat_from_systematics(**params_cov2)


# Your merged indices are in the ORIGINAL (pre-cut) dataset index space
idx_p_merge2 = merged_df2["idx_p"].to_numpy()
idx_d_merge2 = merged_df2["idx_d"].to_numpy()

n_p2 = len(df_p2)
# Extract the proton-proton, deuteron-deuteron, and proton-deuteron sub-blocks:
c_pp2 = cov_full2[:n_p2, :n_p2]  # shape = (Np, Np)
c_dd2 = cov_full2[n_p2:, n_p2:]  # shape = (Nd, Nd)
c_pd2 = cov_full2[:n_p2, n_p2:]  # shape = (Np, Nd)

# Restrict to matched points only (now safe)
c_pp_sub2 = c_pp2[np.ix_(idx_p_merge2, idx_p_merge2)]
c_dd_sub2 = c_dd2[np.ix_(idx_d_merge2, idx_d_merge2)]
c_pd_sub2 = c_pd2[np.ix_(idx_p_merge2, idx_d_merge2)]

# Cov for y = p - d  =>  Cov(y) = Cov(p)+Cov(d)-Cov(p,d)-Cov(d,p)
c_yy2 = c_pp_sub2 + c_dd_sub2 - c_pd_sub2 - c_pd_sub2.T

# Force exact symmetry (numerical)
c_yy2 = 0.5 * (c_yy2 + c_yy2.T)

print("c_yy2 shape:", c_yy2.shape)

# Add jitter until positive-definite
jitter = 1e-6 * float(np.mean(np.diag(c_yy2)))
for _ in range(10):
    try:
        np.linalg.cholesky(c_yy2)
        break
    except np.linalg.LinAlgError:
        c_yy2 = c_yy2 + np.eye(c_yy2.shape[0]) * jitter
        jitter *= 10
else:
    raise RuntimeError("Covariance matrix not positive-definite even after jitter.")

np.save("Dataset/c_yy_newth.npy", c_yy2)


c_yy2 shape: (248, 248)


In [16]:
#Compute reference for closure test, this should be the f0/y0
pdfset2 = lhapdf.getPDFSet("NNPDF40_nnlo_as_01180")  # This PDF can be changed to any toy underlying PDF set
pdf02 = pdfset2.mkPDF(0)
Q02 = fk_p2.Q0
xt3_true2 = np.zeros_like(xgrid2)


# T_3 = (u - ubar) - (d - dbar)
for i, x in enumerate(xgrid2):
    u2 = pdf02.xfxQ(2, x, Q02)
    ub2 = pdf02.xfxQ(-2, x, Q02)
    d2 = pdf02.xfxQ(1, x, Q02)
    db2 = pdf02.xfxQ(-1, x, Q02)
    xt3_true2[i] = (u2 - ub2) - (d2 - db2)


t32 = xt3_true2 / xgrid2 # Or here we can just input the true function directly for T3 or xT3

t3_ref_int2 = np.trapz(xt3_true2 / xgrid2, xgrid2)  # noqa: NPY201


y_theory2 = W2 @ (xt3_true2)  # shape (N,)
y_t3_theory2 = W2 @ (t32)  # shape (N,)

np.save('Dataset/y_theory_newth.npy', y_theory2)
np.save('Dataset/y_t3_theory_newth.npy', y_t3_theory2)
np.save('Dataset/xt3_true_newth.npy', xt3_true2)
np.save('Dataset/t3_true_newth.npy', t32)
print(len(y_theory2))

rng = np.random.default_rng(seed=451)  # you can set seed if you want reproducible “data”
noise2 = rng.multivariate_normal(mean=np.zeros(len(y_theory2)), cov=c_yy2)

y_pseudo2 = y_theory2 + noise2
#y_theory: y0, y_pseudo = y

LHAPDF 6.5.5 loading /opt/homebrew/Caskroom/miniconda/base/envs/nnpdf/share/LHAPDF/NNPDF40_nnlo_as_01180/NNPDF40_nnlo_as_01180_0000.dat
NNPDF40_nnlo_as_01180 PDF set, member #0, version 1; LHAPDF ID = 331100
248
